In [ ]:
!pip -q uninstall -y transformers accelerate bitsandbytes triton peft datasets

!pip -q install \
  "transformers>=4.51.0" \
  "accelerate>=0.34.2" \
  "bitsandbytes>=0.44.1" \
  "peft>=0.13.2" \
  "datasets>=2.20.0" \
  "evaluate" \
  "scikit-learn" \
  "pyarrow"

import transformers, accelerate, bitsandbytes, peft, datasets
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.6 MB/s eta 0:00:00
transformers: 4.57.3
accelerate: 1.12.0
bitsandbytes: 0.49.0
peft: 0.18.0
datasets: 4.4.1


In [ ]:
# === Cell 2: Mount Google Drive ===

from google.colab import drive
drive.mount('/content/drive')


ValueError: mount failed

In [ ]:
# === Cell 3: Imports & basic configuration ===

import os
from collections import Counter

import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [ ]:
# === Cell 3b: Task & model configuration ===

# Hugging Face model: BASE (non-instruct) Qwen2.5-Coder-7B
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B"

# TODO: CHANGE THESE PATHS to where your data actually lives.
# Example if in Drive: "/content/drive/MyDrive/SemEval/taskC/train.parquet"
TRAIN_PATH       = "/content/drive/MyDrive/SemEval_Data/Task_C/train.parquet"
VAL_PATH         = "/content/drive/MyDrive/SemEval_Data/Task_C/validation.parquet"
TEST_PATH        = "/content/drive/MyDrive/SemEval_Data/Task_C/test.parquet"
TEST_SAMPLE_PATH = "/content/drive/MyDrive/SemEval_Data/Task_C/test_sample.parquet"  # 1000 labeled samples for evaluation only

# Task C labels:
# 0: human, 1: ai, 2: hybrid, 3: adversarial
NUM_LABELS = 4

# Training hyperparameters (you can tune these)
MAX_LENGTH = 512         # Increase to 1024 if you have enough VRAM
BATCH_SIZE = 1           # Per-device batch size (small because model is big)
GRAD_ACCUM = 16          # Gradient accumulation steps → effective batch size = BATCH_SIZE * GRAD_ACCUM
LR         = 2e-4        # Learning rate for QLoRA
EPOCHS     = 3

# Where to save model, logs, submission, etc.
OUTPUT_DIR = "/content/qwen2_5_coder_7b_taskC_qlora"  # Consider changing to /content/drive/MyDrive/... to persist across sessions
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Nice human-readable label mapping
id2label = {0: "human", 1: "ai", 2: "hybrid", 3: "adversarial"}
label2id = {v: k for k, v in id2label.items()}


In [ ]:
# === Cell 4: Load SemEval Task C data ===
# Each parquet file is expected to have:
# - code: the code snippet (string)
# - language: programming language (string)
# - generator: generator name or 'human' (string)
# - label: only in train/validation/test_sample (0–3)

train_ds = Dataset.from_parquet(TRAIN_PATH)
val_ds   = Dataset.from_parquet(VAL_PATH)
test_ds  = Dataset.from_parquet(TEST_PATH)
test_sample_ds = Dataset.from_parquet(TEST_SAMPLE_PATH)

print("Train:", train_ds)
print("Val:", val_ds)
print("Test:", test_ds)
print("Test_sample:", test_sample_ds)


In [ ]:
# === Cell 4b: Make sure labels are ints ===
# Some parquet pipelines might store labels as strings; we ensure ints in [0,1,2,3].

def ensure_int_labels(batch):
    if "label" in batch:
        batch["label"] = int(batch["label"])
    return batch

train_ds = train_ds.map(ensure_int_labels)
val_ds   = val_ds.map(ensure_int_labels)
test_sample_ds = test_sample_ds.map(ensure_int_labels)

print("Train label distribution:", Counter(train_ds["label"]))
print("Val label distribution:", Counter(val_ds["label"]))
print("Test_sample label distribution:", Counter(test_sample_ds["label"]))


In [ ]:
# === Cell 5: Load tokenizer ===
# We use the tokenizer corresponding to Qwen2.5-Coder-7B.
# For decoder-only models, we often set the pad_token to eos_token.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


In [ ]:
!nvidia-smi

In [ ]:
print("Train columns:", train_ds.column_names)
print("Val columns:", val_ds.column_names)
print("Test columns:", test_ds.column_names)
print("Test_sample columns:", test_sample_ds.column_names)

In [ ]:
# === Cell 5b: Preprocessing functions ===
# We prepend language and generator info so the model can leverage that metadata.
def preprocess_batch_with_labels(batch):
    """
    For train/val/test_sample (have labels).
    Uses language/generator if present; otherwise falls back to 'unknown'.
    """
    codes = []

    has_lang = "language" in batch
    has_gen  = "generator" in batch

    for i, c in enumerate(batch["code"]):
        lang = batch["language"][i] if has_lang else "unknown"
        gen  = batch["generator"][i] if has_gen else "unknown"
        prefix = f"<lang={lang}> <gen={gen}> "
        codes.append(prefix + c)

    enc = tokenizer(
        codes,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )
    enc["labels"] = batch["label"]
    return enc


def preprocess_batch_test(batch):
    """
    For test set (no labels).
    Uses language/generator if present; otherwise 'unknown'.
    Your test set currently only has: ['ID', 'code', '__index_level_0__'].
    So here we will end up using <lang=unknown> <gen=unknown>.
    """
    codes = []

    has_lang = "language" in batch
    has_gen  = "generator" in batch

    for i, c in enumerate(batch["code"]):
        lang = batch["language"][i] if has_lang else "unknown"
        gen  = batch["generator"][i] if has_gen else "unknown"
        prefix = f"<lang={lang}> <gen={gen}> "
        codes.append(prefix + c)

    enc = tokenizer(
        codes,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )
    return enc

# Apply preprocessing
train_tok = train_ds.map(
    preprocess_batch_with_labels,
    batched=True,
    remove_columns=train_ds.column_names,
)

val_tok = val_ds.map(
    preprocess_batch_with_labels,
    batched=True,
    remove_columns=val_ds.column_names,
)

test_sample_tok = test_sample_ds.map(
    preprocess_batch_with_labels,
    batched=True,
    remove_columns=test_sample_ds.column_names,
)

# For test, we want to drop everything except the ID column
test_tok = test_ds.map(
    preprocess_batch_test,
    batched=True,
    remove_columns=[col for col in test_ds.column_names if col != "ID"],
)

print(train_tok)
print(val_tok)
print(test_sample_tok)
print(test_tok)

Map:   0%|          | 0/900000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 900000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['ID', 'input_ids', 'attention_mask'],
    num_rows: 1000
})


In [ ]:
# === Cell 6: Compute class weights to handle imbalance ===
# This makes the loss pay more attention to rare classes like hybrid & adversarial.

label_counts = Counter(train_ds["label"])
print("Label counts:", label_counts)

total = sum(label_counts.values())
class_weights = []

for i in range(NUM_LABELS):
    freq = label_counts.get(i, 0)
    w = total / (freq + 1e-6)  # inverse frequency
    class_weights.append(w)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
print("Raw class weights:", class_weights)

# Normalize so that average weight = 1 (just for stability)
class_weights = class_weights / class_weights.mean()
print("Normalized class weights:", class_weights)


Label counts: Counter({0: 485483, 1: 210471, 3: 118526, 2: 85520})
Raw class weights: tensor([ 1.8538,  4.2761, 10.5239,  7.5933])
Normalized class weights: tensor([0.3058, 0.7054, 1.7361, 1.2526])


In [ ]:
!pip install -q triton

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
import os
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["BNB_CUDA_VERSION"] = "124" # Update this to match your CUDA version (run nvidia-smi to check)

In [ ]:
# === Cell 7: Load Qwen2.5-Coder-7B with 4-bit quantization (QLoRA) ===
import torch
from transformers import BitsAndBytesConfig, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

device_map = {"": 0}

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map=device_map,
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


PackageNotFoundError: No package metadata was found for bitsandbytes

In [ ]:
# === Cell 8: Metrics function ===
# Trainer will call this during evaluation to compute macro/micro F1.

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    micro_f1 = f1_score(labels, preds, average="micro")
    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
    }


In [ ]:
# === Cell 9: Custom Trainer to apply class-weighted loss ===

class WeightedTrainer(Trainer):
    """
    A custom Trainer that overrides compute_loss to use class weights
    in the cross-entropy loss function. This helps with imbalanced classes.
    """
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = torch.nn.CrossEntropyLoss()

        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1),
        )

        if return_outputs:
            return loss, outputs
        return loss


In [ ]:
# === Cell 10: TrainingArguments and Trainer ===

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.05,
    logging_steps=50,

    # Transformers 4.46 syntax:
    evaluation_strategy="epoch",    # evaluate at end of each epoch
    save_strategy="epoch",          # save checkpoint each epoch

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,        # <-- valid in 4.46, no warning
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [ ]:
# === Cell 11: Train the model ===

train_result = trainer.train()

# Save final adapter and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training finished.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)


In [ ]:
# === Cell 12: Detailed evaluation on validation and test_sample ===

def evaluate_split(name, tokenized_ds, original_ds):
    """
    Helper to:
    - run predictions on a dataset
    - compute macro/micro F1
    - print per-class metrics and confusion matrix
    """
    print(f"\n===== Evaluation on {name} =====")
    pred_output = trainer.predict(tokenized_ds)
    logits = pred_output.predictions
    labels = np.array(original_ds["label"])
    preds = logits.argmax(axis=-1)

    macro_f1 = f1_score(labels, preds, average="macro")
    micro_f1 = f1_score(labels, preds, average="micro")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Micro F1: {micro_f1:.4f}\n")

    print("Classification report:")
    print(classification_report(labels, preds, target_names=[id2label[i] for i in range(NUM_LABELS)]))

    print("Confusion matrix (rows = true, cols = pred):")
    print(confusion_matrix(labels, preds))


# Evaluate on validation set
evaluate_split("validation", val_tok, val_ds)

# Evaluate on test_sample (1k labeled sample; not used in final official scoring)
evaluate_split("test_sample", test_sample_tok, test_sample_ds)


In [ ]:
# === Cell 13: Generate Kaggle submission on test.parquet ===

# Run predictions on the test set (unlabeled)
test_logits = trainer.predict(test_tok).predictions
test_pred_ids = test_logits.argmax(axis=-1)

# Use 'ID' column (capital I, D) from test_ds
if "ID" in test_ds.column_names:
    ids = test_ds["ID"]
else:
    ids = list(range(len(test_ds)))  # fallback, but in your case we DO have "ID"

submission = pd.DataFrame({
    "id": ids,                      # Kaggle expects column name 'id' in lowercase
    "label": test_pred_ids.astype(int),
})

submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
submission.head()
